In [7]:
import os
import glob
import pandas as pd
import numpy as np

base_path = r"D:\FallDetection Research\Sample_Training" 
file_pattern = os.path.join(base_path, "**", "*.csv")
all_files = glob.glob(file_pattern, recursive=True)

data_frames = []
window_size = 50 # 50 rows = approx 0.5 seconds

for file_path in all_files:
    # 1. Ek single file load karo
    df = pd.read_csv(file_path)
    
    # 2. Usi file ke andar basic features banao
    df['Acc_SVM'] = np.sqrt(df['AccX']**2 + df['AccY']**2 + df['AccZ']**2)
    df['Gyr_SVM'] = np.sqrt(df['GyrX']**2 + df['GyrY']**2 + df['GyrZ']**2)
    df['Tilt_Angle'] = np.sqrt(df['EulerX']**2 + df['EulerY']**2)
    
    # 3. YAHAN PAR SLIDING WINDOW LAGAO (Sirf isi current file ke data par)
    df['Acc_Memory'] = df['Acc_SVM'].rolling(window=window_size, min_periods=1).max()
    df['Gyr_Memory'] = df['Gyr_SVM'].rolling(window=window_size, min_periods=1).max()
    df['Tilt_Memory'] = df['Tilt_Angle'].rolling(window=window_size, min_periods=1).max()
    
    # 4. Ab is perfectly processed file ko list mein daal do
    data_frames.append(df)

# 5. Ab sabko merge karo! Koi boundary overlap nahi hoga.
master_dataset = pd.concat(data_frames, ignore_index=True)

print("Data loaded and Memory Features calculated perfectly!")

Data loaded and Memory Features calculated perfectly!


In [8]:
master_dataset.head()

,TimeStamp(s),FrameCounter,AccX,AccY,AccZ,GyrX,GyrY,GyrZ,EulerX,EulerY,EulerZ,FallCheck,Acc_SVM,Gyr_SVM,Tilt_Angle,Acc_Memory,Gyr_Memory,Tilt_Memory
0,0.01,1,-0.058995,-1.006997,-0.050006,-3.608840,-7.162425,0.859354,87.926101,-3.093978,-53.124675,0,1.009962,8.066136,87.980520,1.009962,8.066136,87.98052
1,0.02,2,-0.055983,-1.002583,-0.059927,-4.981095,-5.470213,0.849338,87.878890,-3.101371,-53.173120,0,1.005932,7.446873,87.933599,1.009962,8.066136,87.98052
2,0.03,3,-0.056324,-0.999016,-0.067955,-5.703384,-4.568670,0.877325,87.823490,-3.110706,-53.219636,0,1.002907,7.360097,87.878563,1.009962,8.066136,87.98052
3,0.04,4,-0.059694,-0.997563,-0.071426,-5.632291,-4.281338,0.991394,87.759410,-3.122727,-53.261463,0,1.001896,7.143908,87.814950,1.009962,8.066136,87.98052
4,0.05,5,-0.062891,-0.999523,-0.069458,-5.194557,-3.702469,1.203265,87.694780,-3.137237,-53.294808,0,1.003905,6.491498,87.750879,1.009962,8.066136,87.98052


In [9]:
import numpy as np
from sklearn.metrics import f1_score

# Threshold ranges (Wahi pehle wali)
T1_range = np.arange(1.5, 4.0, 0.5) 
T2_range = np.arange(50, 300, 50)   
T3_range = np.arange(30, 80, 10)    

best_f1 = 0
best_thresholds = (0, 0, 0)

print("Memory Features par Grid Search start ho raha hai...")

for t1 in T1_range:
    for t2 in T2_range:
        for t3 in T3_range:
            
            # DHYAN DEIN: Yahan hum ab '_Memory' columns use kar rahe hain
            predictions = (
                (master_dataset['Acc_Memory'] > t1) & 
                (master_dataset['Gyr_Memory'] > t2) & 
                (master_dataset['Tilt_Memory'] > t3)
            ).astype(int)
            
            # Agar tumhare target column ka naam 'Label' hai toh ise change kar lena
            target_col = master_dataset['FallCheck'] 
            
            # F1-Score calculate karna
            score = f1_score(target_col, predictions, zero_division=0)
            
            if score > best_f1:
                best_f1 = score
                best_thresholds = (t1, t2, t3)

print("\n--- Calibration Complete! ---")
print(f"Optimal T1 (Acc_Memory): {best_thresholds[0]}")
print(f"Optimal T2 (Gyr_Memory): {best_thresholds[1]}")
print(f"Optimal T3 (Tilt_Memory): {best_thresholds[2]}")
print(f"Best F1-Score on Training Data: {best_f1:.4f}")

Memory Features par Grid Search start ho raha hai...

--- Calibration Complete! ---
Optimal T1 (Acc_Memory): 1.5
Optimal T2 (Gyr_Memory): 150
Optimal T3 (Tilt_Memory): 30
Best F1-Score on Training Data: 0.3345


In [10]:
import pandas as pd

def event_based_evaluation(y_true, y_pred):
    # 1. 'Event' IDs banana: Jab bhi true label 0 se 1 ya 1 se 0 mein badle, usko ek naya event maano
    events = (y_true != y_true.shift()).cumsum()
    
    # Ek temporary dataframe banate hain calculations ke liye
    df_eval = pd.DataFrame({'Actual': y_true, 'Predicted': y_pred, 'Event_ID': events})
    
    # 2. Har event (task) ki summary nikalna
    # Aggregation logic: Agar us poore event mein ek baar bhi 1 predict hua hai, toh usko 1 maan lo
    event_summary = df_eval.groupby('Event_ID').max()
    
    TP = FP = TN = FN = 0
    
    # 3. Har event ko check karke TP, TN, FP, FN mein daalna
    for index, row in event_summary.iterrows():
        actual = row['Actual']
        predicted = row['Predicted']
        
        if actual == 1: # Agar yeh actual Fall event tha
            if predicted == 1:
                TP += 1  # Airbag successfully khula!
            else:
                FN += 1  # Miss ho gaya!
        else:           # Agar yeh Normal Activity thi
            if predicted == 1:
                FP += 1  # False Alarm!
            else:
                TN += 1  # Shanti rahi (Correct)
                
    # 4. Final Metrics Calculate Karna
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    sensitivity = TP / (TP + FN) if (TP + FN) > 0 else 0
    accuracy = (TP + TN) / (TP + TN + FP + FN) if (TP + TN + FP + FN) > 0 else 0
    f1 = 2 * (precision * sensitivity) / (precision + sensitivity) if (precision + sensitivity) > 0 else 0
    
    # Print the reality
    print("--- 🚀 RESEARCH-GRADE EVENT-BASED METRICS 🚀 ---")
    print(f"Total Actual Fall Events: {TP + FN}")
    print(f"Total Normal Events: {TN + FP}")
    print("-" * 45)
    print(f"True Positives (Sahi pakde gaye Falls): {TP}")
    print(f"False Negatives (Miss ho gaye Falls): {FN}")
    print(f"False Positives (Galtiyan / False Alarms): {FP}")
    print(f"True Negatives (Sahi Normal Activities): {TN}")
    print("-" * 45)
    print(f"Sensitivity (Recall): {sensitivity * 100:.2f}%")
    print(f"Precision: {precision * 100:.2f}%")
    print(f"Accuracy: {accuracy * 100:.2f}%")
    print(f"Final Event-Based F1-Score: {f1 * 100:.2f}%")
    
    return f1

# Ab is function mein apne data aur best thresholds wali predictions pass karte hain
# (Optimal thresholds jo hume pichle step mein mile the: 1.5, 150, 30)
final_predictions = (
    (master_dataset['Acc_Memory'] > 1.5) & 
    (master_dataset['Gyr_Memory'] > 150) & 
    (master_dataset['Tilt_Memory'] > 30)
).astype(int)

# Execute the function!
event_based_f1 = event_based_evaluation(master_dataset['FallCheck'], final_predictions)

--- 🚀 RESEARCH-GRADE EVENT-BASED METRICS 🚀 ---
Total Actual Fall Events: 1862
Total Normal Events: 1862
---------------------------------------------
True Positives (Sahi pakde gaye Falls): 1799
False Negatives (Miss ho gaye Falls): 63
False Positives (Galtiyan / False Alarms): 75
True Negatives (Sahi Normal Activities): 1787
---------------------------------------------
Sensitivity (Recall): 96.62%
Precision: 96.00%
Accuracy: 96.29%
Final Event-Based F1-Score: 96.31%


In [11]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score

print("Data ko Events mein compress kar rahe hain...")

# 1. Dataset ko Events mein todna (Jaisa evaluation function mein kiya tha)
master_dataset['Event_ID'] = (master_dataset['FallCheck'] != master_dataset['FallCheck'].shift()).cumsum()

# 2. Har Event ka ek single summary row banana
# Agar threshold is max value ko cross kar gaya, matlab us event mein alarm baj jayega
event_df = master_dataset.groupby('Event_ID').agg({
    'FallCheck': 'max',     # Yeh batayega ki yeh event Fall(1) tha ya Normal(0)
    'Acc_Memory': 'max',    # Us event mein Acc ka sabse bada peak
    'Gyr_Memory': 'max',    # Us event mein Gyr ka sabse bada peak
    'Tilt_Memory': 'max'    # Us event mein Tilt ka sabse bada peak
}).reset_index()

print(f"Total Events extracted: {len(event_df)} (Falls: {event_df['FallCheck'].sum()}, Normal: {len(event_df) - event_df['FallCheck'].sum()})")

# 3. Stratified K-Fold Setup (k=5)
# Yeh ensure karega ki har fold mein Falls aur Normal ka ratio same rahe
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Threshold ranges
T1_range = np.arange(1.5, 4.0, 0.5)
T2_range = np.arange(50, 300, 50)
T3_range = np.arange(30, 80, 10)

fold = 1
cv_f1_scores = []

print("\n🚀 Starting 5-Fold Stratified Cross-Validation 🚀\n")

for train_idx, test_idx in skf.split(event_df, event_df['FallCheck']):
    # Train aur Test events alag karna
    train_events = event_df.iloc[train_idx]
    test_events = event_df.iloc[test_idx]
    
    best_f1 = 0
    best_thresh = (0,0,0)
    
    # 4. Grid Search ONLY on Train Fold
    for t1 in T1_range:
        for t2 in T2_range:
            for t3 in T3_range:
                # Train fold pe prediction
                preds = ((train_events['Acc_Memory'] > t1) & 
                         (train_events['Gyr_Memory'] > t2) & 
                         (train_events['Tilt_Memory'] > t3)).astype(int)
                
                # F1 calculate karna
                score = f1_score(train_events['FallCheck'], preds, zero_division=0)
                if score > best_f1:
                    best_f1 = score
                    best_thresh = (t1, t2, t3)
                    
    # 5. Evaluate Best Thresholds on Unseen TEST Fold
    test_preds = ((test_events['Acc_Memory'] > best_thresh[0]) & 
                  (test_events['Gyr_Memory'] > best_thresh[1]) & 
                  (test_events['Tilt_Memory'] > best_thresh[2])).astype(int)
                  
    test_f1 = f1_score(test_events['FallCheck'], test_preds, zero_division=0)
    cv_f1_scores.append(test_f1)
    
    print(f"Fold {fold}: Best Thresh = {best_thresh} | Train F1 = {best_f1:.4f} | TEST F1 = {test_f1:.4f}")
    fold += 1

# 6. Final Results
print("-" * 50)
print(f"🏆 Average Cross-Validation F1-Score: {np.mean(cv_f1_scores)*100:.2f}%")
print("-" * 50)

Data ko Events mein compress kar rahe hain...
Total Events extracted: 3724 (Falls: 1862, Normal: 1862)

🚀 Starting 5-Fold Stratified Cross-Validation 🚀

Fold 1: Best Thresh = (np.float64(2.5), np.int64(150), np.int64(30)) | Train F1 = 0.9809 | TEST F1 = 0.9733
Fold 2: Best Thresh = (np.float64(2.5), np.int64(150), np.int64(30)) | Train F1 = 0.9796 | TEST F1 = 0.9786
Fold 3: Best Thresh = (np.float64(2.5), np.int64(150), np.int64(30)) | Train F1 = 0.9789 | TEST F1 = 0.9813
Fold 4: Best Thresh = (np.float64(2.5), np.int64(150), np.int64(30)) | Train F1 = 0.9799 | TEST F1 = 0.9773
Fold 5: Best Thresh = (np.float64(2.5), np.int64(150), np.int64(30)) | Train F1 = 0.9776 | TEST F1 = 0.9865
--------------------------------------------------
🏆 Average Cross-Validation F1-Score: 97.94%
--------------------------------------------------
